# Lab 10 — Prompt Chaining with Verification Gates
**Pattern: PROMPT CHAINING** — from Anthropic's five agentic patterns

---

## What is Prompt Chaining?

Break a complex task into a **sequence of focused steps**.  
Each step does exactly one job and passes its output to the next.  
Between each step, a **gate** checks the output before the next step runs.

If a gate fails — stop. Don't waste tokens running Step 3 on broken Step 2 output.

---

## Real-world analogy

**A CI/CD pipeline.**

```
Build → Unit tests → Integration tests → Security scan → Deploy
```

Each stage does one thing. Each stage is verifiable.  
If unit tests fail, the pipeline stops — you don't deploy broken code.  
A gate that fails early saves you from damage downstream.

This lab builds the same pattern for document generation:

```
Extract requirements → Generate outline → Write document → Polish
       Gate 1               Gate 2            Gate 3
```

---

## Why not one big prompt?

A single mega-prompt trying to do all four steps simultaneously:
- Cannot be gated (you can't verify partial output)
- Tends to skip or compress steps
- Is harder to debug when something is wrong
- Cannot restart from a checkpoint — it's all or nothing

Chaining gives you checkpoints, verifiability, and restartability.

---

## What you will learn
1. How to decompose a complex task into focused, verifiable steps
2. How programmatic gates protect downstream steps from bad input
3. Why gates should be code, not LLM calls
4. How to pass structured output from one step to the next safely
5. When to abort vs. when to continue

## Step 1 — Setup

In [ ]:
import anthropic
import json
import re
from dataclasses import dataclass, field
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic()
print("✓ Client ready")

## Step 2 — Pipeline Data Structures

Two simple dataclasses to track results across the pipeline.
`StepResult` records what happened at each step.  
`PipelineResult` holds all steps and the final output.

In [ ]:
@dataclass
class StepResult:
    step_name:  str
    output:     str
    passed:     bool
    gate_notes: str = ""

@dataclass
class PipelineResult:
    steps:      list = field(default_factory=list)
    final:      str  = ""
    success:    bool = False
    aborted_at: str  = ""

    def add(self, result: StepResult):
        self.steps.append(result)

    def show_summary(self):
        print("\n── Pipeline Execution Summary ──")
        for s in self.steps:
            icon = "✓" if s.passed else "✗"
            print(f"  [{icon}] {s.step_name}")
            if s.gate_notes:
                print(f"       Gate: {s.gate_notes}")
        if self.success:
            print(f"\n  ✓ Pipeline completed successfully")
        else:
            print(f"  ✗ Pipeline aborted at: {self.aborted_at}")

print("✓ Data structures defined")

## Step 3 — Verification Gates

Gates are **programmatic checks — not LLM calls**.

> LLMs are good at generation.  
> Programs are better at validating structure, format, and completeness.  
> Use each for what it's good at.

Three gates, one per inter-step boundary:
- **Gate 1:** Did Step 1 return valid JSON with all required fields?
- **Gate 2:** Does the outline have Introduction, Conclusion, and 3+ sections?
- **Gate 3:** Is the document 150+ words with structural headers?

In [ ]:
def gate_requirements(output: str) -> tuple:
    """Gate 1: All required fields present in the JSON extraction?"""
    required = ["audience", "purpose", "key_topics", "format", "length"]
    try:
        data    = json.loads(output)
        missing = [f for f in required if not data.get(f)]
        if missing:
            return False, f"Missing required fields: {missing}"
        topics = data.get("key_topics", [])
        if not isinstance(topics, list) or len(topics) < 2:
            return False, f"key_topics must be a list with 2+ items. Got: {topics}"
        return True, f"All {len(required)} fields present. {len(topics)} topics identified."
    except json.JSONDecodeError as e:
        return False, f"Invalid JSON: {e}"


def gate_outline(output: str) -> tuple:
    """Gate 2: Outline has required sections?"""
    lower   = output.lower()
    missing = [s for s in ["introduction", "conclusion"] if s not in lower]
    if missing:
        return False, f"Missing required sections: {missing}"
    sections = [l for l in output.split("\n")
                if l.strip().startswith(("##", "**", "2.", "3.", "4."))]
    if len(sections) < 3:
        return False, f"Only {len(sections)} sections found. Minimum 3 required."
    return True, f"Outline valid — {len(sections)} sections detected."


def gate_document(output: str) -> tuple:
    """Gate 3: Document is long enough and has structure?"""
    word_count  = len(output.split())
    if word_count < 150:
        return False, f"Too short: {word_count} words. Minimum 150 required."
    has_headers = bool(re.search(r'#{1,3}\s+\w+|^\*\*\w+', output, re.MULTILINE))
    if not has_headers:
        return False, "No headers detected. Document needs structure."
    return True, f"Valid — {word_count} words with headers."

print("✓ Gates defined: gate_requirements, gate_outline, gate_document")

## Step 4 — Pipeline Steps

Four focused steps. Each does **one job**.

> Step 3 receives outputs from **both** Step 1 AND Step 2.  
> This is not a strict linear chain — outputs from any previous step  
> can be passed to any later step. It's a dependency graph.

In [ ]:
def step_extract_requirements(brief: str) -> str:
    """Step 1: Extract structured requirements from free-text brief. Output: JSON."""
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=512,
        system="""You are a requirements analyst. Extract structured requirements.
Return ONLY a JSON object:
{
  "audience":   "who will read this",
  "purpose":    "what the document must achieve",
  "key_topics": ["topic 1", "topic 2", "topic 3"],
  "format":     "report/guide/brief/etc",
  "length":     "short/medium/long",
  "tone":       "professional/technical/accessible"
}
Return ONLY valid JSON.""",
        messages=[{"role": "user", "content": f"Extract requirements from:\n\n{brief}"}]
    )
    raw = response.content[0].text.strip()
    if raw.startswith("```"):
        raw = "\n".join(raw.split("\n")[1:-1])
    return raw


def step_generate_outline(requirements_json: str) -> str:
    """Step 2: Generate a structured outline using the requirements JSON."""
    try:
        req = json.loads(requirements_json)
    except:
        req = {}
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=512,
        system=f"""You are a document architect. Create a clear outline.
Audience: {req.get('audience', 'general')}
Topics to cover: {', '.join(req.get('key_topics', []))}
Include ## Introduction, ## sections per topic, ## Conclusion.
Add 2-3 bullet points per section showing content.""",
        messages=[{"role": "user", "content": "Create an outline for these requirements."}]
    )
    return response.content[0].text


def step_write_document(outline: str, requirements_json: str) -> str:
    """
    Step 3: Write the full document.
    Receives outputs from BOTH Step 1 (requirements) AND Step 2 (outline).
    """
    try:
        req = json.loads(requirements_json)
    except:
        req = {}
    length_guide = {"short": "~300 words", "medium": "~500 words", "long": "~800 words"}
    target = length_guide.get(req.get("length", "medium"), "~500 words")
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1500,
        system=f"""You are a technical writer.
Audience: {req.get('audience', 'general')}
Purpose:  {req.get('purpose', 'inform')}
Tone:     {req.get('tone', 'professional')}
Target:   {target}
Follow the outline exactly. Use markdown headers. Be specific — no generic statements.""",
        messages=[{"role": "user", "content": f"Write the full document following:\n\n{outline}"}]
    )
    return response.content[0].text


def step_polish(draft: str, requirements_json: str) -> str:
    """Step 4: Light editorial polish. Fix flow, tone, opening/closing. Don't change structure."""
    try:
        audience = json.loads(requirements_json).get("audience", "general readers")
    except:
        audience = "general readers"
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1500,
        system=f"""You are a senior editor polishing for {audience}.
Fix awkward sentences, inconsistent tone, weak opening/closing.
Do NOT change structure, headings, or key content.
Return the complete polished document.""",
        messages=[{"role": "user", "content": f"Polish this document:\n\n{draft}"}]
    )
    return response.content[0].text

print("✓ All four steps defined")

## Step 5 — The Full Pipeline

Ties everything together. Gates run between each step. Abort early if any gate fails.

In [ ]:
def run_pipeline(brief: str) -> PipelineResult:
    """
    Four-step pipeline with programmatic gates.
    If any gate fails, the pipeline aborts — no wasted tokens on broken input.
    """
    result = PipelineResult()
    print(f"\n── Brief: {brief[:80].strip()}...")
    print("─" * 60)

    # STEP 1 ──────────────────────────────────────────────────────
    print("\n[STEP 1] Extracting requirements...")
    out1   = step_extract_requirements(brief)
    ok, notes = gate_requirements(out1)
    result.add(StepResult("Step 1: Extract Requirements", out1, ok, notes))
    print(f"  Gate 1: {'PASS ✓' if ok else 'FAIL ✗'} — {notes}")
    if not ok:
        result.aborted_at = "Step 1 Gate"
        return result

    # STEP 2 ──────────────────────────────────────────────────────
    print("\n[STEP 2] Generating outline...")
    out2   = step_generate_outline(out1)
    ok, notes = gate_outline(out2)
    result.add(StepResult("Step 2: Generate Outline", out2, ok, notes))
    print(f"  Gate 2: {'PASS ✓' if ok else 'FAIL ✗'} — {notes}")
    if not ok:
        result.aborted_at = "Step 2 Gate"
        return result

    # STEP 3 ──────────────────────────────────────────────────────
    print("\n[STEP 3] Writing document...")
    out3   = step_write_document(out2, out1)   # ← uses Step 1 AND Step 2 outputs
    ok, notes = gate_document(out3)
    result.add(StepResult("Step 3: Write Document", out3, ok, notes))
    print(f"  Gate 3: {'PASS ✓' if ok else 'FAIL ✗'} — {notes}")
    if not ok:
        result.aborted_at = "Step 3 Gate"
        return result

    # STEP 4 ──────────────────────────────────────────────────────
    print("\n[STEP 4] Polishing...")
    out4   = step_polish(out3, out1)
    result.add(StepResult("Step 4: Polish", out4, True, "Final step — no gate"))
    result.final   = out4
    result.success = True
    return result

print("✓ Pipeline defined")

## Step 6 — Run the Pipeline

In [ ]:
brief = """
I need a technical guide for mid-level cloud engineers evaluating
whether to move from self-managed Kubernetes to EKS.
It should be professional but accessible, cover tradeoffs, operational
implications, and cost considerations.
Aim for something they can read in 5 minutes.
"""

pipeline_result = run_pipeline(brief)

In [ ]:
pipeline_result.show_summary()

In [ ]:
if pipeline_result.success:
    print("\n" + "=" * 60)
    print("FINAL DOCUMENT:")
    print("=" * 60)
    print(pipeline_result.final)

## Key Takeaways

| Point | Why it matters |
|-------|---------------|
| Each step does ONE job | Focus makes output verifiable. A mega-prompt cannot be gated. |
| Gates are code, not LLM calls | Programs validate structure reliably. LLMs would add ambiguity. |
| Abort early on gate failure | Stop before wasting tokens on broken downstream steps. Same as a failing CI stage. |
| Structured output between steps | JSON from Step 1 is parsed reliably. Prose would force interpretation. |
| Later steps can use any earlier output | Step 3 receives Step 1 AND Step 2. It's a dependency graph, not just a chain. |

---

## You have now completed all ten labs

| Lab | Pattern | HTML Section |
|-----|---------|-------------|
| Lab 1 | First API call | 12 |
| Lab 2 | Memory agent | 02 |
| Lab 3 | Tool chaining | 07 |
| Lab 4 | Multi-agent (supervisor) | 03, 07 |
| Lab 5 | Routing | 07 |
| Lab 6 | FSM-controlled agent | 04 |
| Lab 7 | Evaluator-Optimizer | 07 |
| Lab 8 | MCP-style tool registry | 08 |
| Lab 9 | Orchestrator fan-out | 03, 07 |
| Lab 10 | Prompt chaining with gates | 07 |